# 🏦 Bank Customer Churn Prediction
## Daily Challenge — Machine Learning Classification

**Objectif :** Prédire si un client va quitter la banque (churn) en utilisant ses données personnelles, financières et comportementales.

---

## 📦 Section 1 : Setup & Data Loading

In [ ]:
# Installation des librairies
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    accuracy_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'churn': '#E24B4A', 'no_churn': '#1D9E75', 'blue': '#185FA5', 'purple': '#534AB7'}
print('✅ Librairies importées avec succès')

In [ ]:
# === CHARGEMENT DU DATASET ===
# TODO : Charger le dataset Bank Customer Churn

try:
    # Tentative depuis le zip GitHub
    import io, zipfile, requests
    url = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Inferential%20Statistics.zip'
    r = requests.get(url, timeout=15)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    # Chercher le fichier churn dans le zip
    churn_file = [f for f in z.namelist() if 'churn' in f.lower() or 'bank' in f.lower()]
    if churn_file:
        df = pd.read_csv(z.open(churn_file[0]))
        print(f'✅ Dataset chargé depuis GitHub : {churn_file[0]}')
    else:
        raise FileNotFoundError('Fichier churn non trouvé dans le zip')
except Exception as e:
    print(f'⚠️  Chargement GitHub échoué ({e})')
    print('   Génération de données simulées réalistes...')

    np.random.seed(42)
    n = 10000

    credit_score    = np.random.randint(300, 850, n)
    country         = np.random.choice(['France', 'Germany', 'Spain'], n, p=[0.5, 0.25, 0.25])
    gender          = np.random.choice(['Male', 'Female'], n)
    age             = np.random.randint(18, 92, n)
    tenure          = np.random.randint(0, 11, n)
    balance         = np.where(np.random.rand(n) < 0.3, 0,
                               np.random.uniform(1000, 250000, n))
    num_products    = np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.45, 0.03, 0.02])
    has_cr_card     = np.random.choice([0, 1], n, p=[0.3, 0.7])
    is_active       = np.random.choice([0, 1], n, p=[0.5, 0.5])
    estimated_salary = np.random.uniform(10000, 200000, n)

    # Churn simulé avec des règles réalistes
    churn_prob = (
        0.08
        + 0.15 * (age > 55).astype(int)
        + 0.10 * (balance == 0).astype(int)
        + 0.20 * (num_products >= 3).astype(int)
        - 0.10 * is_active
        - 0.05 * (tenure > 5).astype(int)
        + 0.05 * (country == 'Germany').astype(int)
    )
    churn_prob = np.clip(churn_prob, 0.02, 0.90)
    exited = (np.random.rand(n) < churn_prob).astype(int)

    df = pd.DataFrame({
        'CustomerId':       np.arange(15000001, 15000001 + n),
        'Surname':          ['Customer_' + str(i) for i in range(n)],
        'CreditScore':      credit_score,
        'Geography':        country,
        'Gender':           gender,
        'Age':              age,
        'Tenure':           tenure,
        'Balance':          balance.round(2),
        'NumOfProducts':    num_products,
        'HasCrCard':        has_cr_card,
        'IsActiveMember':   is_active,
        'EstimatedSalary':  estimated_salary.round(2),
        'Exited':           exited
    })

    # Ajout de quelques valeurs manquantes réalistes
    for col in ['CreditScore', 'Age', 'Balance']:
        idx = np.random.choice(n, 30, replace=False)
        df.loc[idx, col] = np.nan

    print(f'✅ Dataset simulé : {n} clients')

print(f'\n📐 Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()

---
## 🔍 Section 2 : Exploratory Data Analysis (EDA)

In [ ]:
# TODO : Explorer le dataset
print('=== INFORMATIONS GÉNÉRALES ===')
df.info()
print('\n=== STATISTIQUES DESCRIPTIVES ===')
df.describe().round(2)

In [ ]:
# TODO : Vérifier les valeurs manquantes
print('=== VALEURS MANQUANTES ===')
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Count': missing, '%': pct})[missing > 0])

print('\n=== DISTRIBUTION DE LA VARIABLE CIBLE ===')
churn_counts = df['Exited'].value_counts()
print(churn_counts)
print(f'Taux de churn : {churn_counts[1]/len(df)*100:.1f}%')

In [ ]:
# TODO : Visualisations EDA
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Distribution churn
labels = ['Resté (0)', 'Churné (1)']
vals   = [churn_counts.get(0, 0), churn_counts.get(1, 0)]
axes[0,0].bar(labels, vals, color=[COLORS['no_churn'], COLORS['churn']], edgecolor='white', width=0.5)
axes[0,0].set_title('Distribution Churn', fontweight='bold')
for i, v in enumerate(vals):
    axes[0,0].text(i, v/2, f'{v:,}\n({v/sum(vals)*100:.1f}%)', ha='center',
                   color='white', fontweight='bold', fontsize=12)

# 2. Age vs Churn
df.groupby('Exited')['Age'].plot.kde(ax=axes[0,1],
    color=[COLORS['no_churn'], COLORS['churn']])
axes[0,1].set_title('Distribution de l\'Âge par Churn', fontweight='bold')
axes[0,1].legend(['Resté', 'Churné'])
axes[0,1].set_xlabel('Âge')

# 3. Balance vs Churn
df[df['Exited']==0]['Balance'].plot.hist(ax=axes[0,2], bins=30, alpha=0.6,
    color=COLORS['no_churn'], label='Resté', density=True)
df[df['Exited']==1]['Balance'].plot.hist(ax=axes[0,2], bins=30, alpha=0.6,
    color=COLORS['churn'], label='Churné', density=True)
axes[0,2].set_title('Distribution Balance par Churn', fontweight='bold')
axes[0,2].legend(); axes[0,2].set_xlabel('Balance')

# 4. NumOfProducts vs Churn
ct = pd.crosstab(df['NumOfProducts'], df['Exited'], normalize='index') * 100
ct.plot(kind='bar', ax=axes[1,0], color=[COLORS['no_churn'], COLORS['churn']],
        edgecolor='white')
axes[1,0].set_title('Taux de churn par Nb produits', fontweight='bold')
axes[1,0].set_xlabel('Nombre de produits'); axes[1,0].set_ylabel('%')
axes[1,0].tick_params(axis='x', rotation=0)
axes[1,0].legend(['Resté', 'Churné'])

# 5. Geography vs Churn
if 'Geography' in df.columns:
    ct2 = pd.crosstab(df['Geography'], df['Exited'], normalize='index') * 100
    ct2.plot(kind='bar', ax=axes[1,1], color=[COLORS['no_churn'], COLORS['churn']],
             edgecolor='white')
    axes[1,1].set_title('Taux de churn par Pays', fontweight='bold')
    axes[1,1].tick_params(axis='x', rotation=30)
    axes[1,1].legend(['Resté', 'Churné'])

# 6. IsActiveMember vs Churn
ct3 = pd.crosstab(df['IsActiveMember'], df['Exited'], normalize='index') * 100
ct3.index = ['Inactif', 'Actif']
ct3.plot(kind='bar', ax=axes[1,2], color=[COLORS['no_churn'], COLORS['churn']],
         edgecolor='white')
axes[1,2].set_title('Taux de churn : Actif vs Inactif', fontweight='bold')
axes[1,2].tick_params(axis='x', rotation=0)
axes[1,2].legend(['Resté', 'Churné'])

plt.suptitle('Exploratory Data Analysis — Bank Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de corrélation (variables numériques)
num_cols = df.select_dtypes(include='number').columns.tolist()
# Exclure identifiants
num_cols = [c for c in num_cols if c not in ['CustomerId', 'RowNumber']]

corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5, square=True)
ax.set_title('Matrice de Corrélation', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Corrélations avec Exited
print('\n🎯 Corrélations avec Exited (churn) :')
corr_exited = corr['Exited'].drop('Exited').sort_values(key=abs, ascending=False)
for feat, val in corr_exited.items():
    bar = '█' * int(abs(val) * 20)
    print(f'  {feat:20s} : {val:+.3f} {bar}')

---
## 🛠️ Section 3 : Data Preprocessing

In [ ]:
# TODO : Preprocessing complet
df_clean = df.copy()

# 1. Suppression des colonnes non pertinentes
cols_to_drop = [c for c in ['CustomerId', 'RowNumber', 'Surname'] if c in df_clean.columns]
df_clean.drop(columns=cols_to_drop, inplace=True)
print(f'✅ Colonnes supprimées : {cols_to_drop}')

# 2. Gestion des valeurs manquantes
for col in df_clean.select_dtypes(include='number').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
        print(f'✅ {col} : NaN → médiane')

# 3. Encodage des variables catégorielles
le = LabelEncoder()
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = le.fit_transform(df_clean[col])
    print(f'✅ {col} encodé (LabelEncoder)')

# 4. Feature Engineering
df_clean['BalanceToSalary']    = df_clean['Balance'] / (df_clean['EstimatedSalary'] + 1)
df_clean['AgeGroup']           = pd.cut(df_clean['Age'],
                                          bins=[0,30,45,60,100],
                                          labels=[0,1,2,3]).astype(int)
df_clean['CreditScoreGroup']   = pd.cut(df_clean['CreditScore'],
                                          bins=[0,579,669,739,799,900],
                                          labels=[0,1,2,3,4]).astype(int)
df_clean['ZeroBalance']        = (df_clean['Balance'] == 0).astype(int)
df_clean['ProductsPerTenure']  = df_clean['NumOfProducts'] / (df_clean['Tenure'] + 1)
print('\n✅ Feature engineering terminé')

print(f'\n📐 Dataset final : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes')
print(f'Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}')
df_clean.head()

In [ ]:
# TODO : Split et standardisation
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Gestion du déséquilibre avec SMOTE (sur train uniquement)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print(f'Train original  : {X_train.shape[0]:,} | Churn : {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Train rééch.    : {X_train_res.shape[0]:,} | Churn : {y_train_res.sum():,} ({y_train_res.mean()*100:.1f}%)')
print(f'Test            : {X_test.shape[0]:,} | Churn : {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

---
## 🤖 Section 4 : Model Training & Evaluation

In [ ]:
# TODO : Entraîner et évaluer plusieurs modèles
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42,
                                          eval_metric='logloss', verbosity=0)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_scaled, y_train_res)
    y_pred  = model.predict(X_test_scaled)
    y_prob  = model.predict_proba(X_test_scaled)[:, 1]
    cv_f1   = cross_val_score(model, X_train_scaled, y_train_res, cv=cv, scoring='f1')

    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'cv_f1':     cv_f1.mean(),
        'cv_std':    cv_f1.std(),
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'model':     model
    }
    print(f'✅ {name} entraîné')

# Tableau comparatif
comp = pd.DataFrame({
    n: {
        'Accuracy':    f"{r['accuracy']:.3f}",
        'Precision':   f"{r['precision']:.3f}",
        'Recall':      f"{r['recall']:.3f}",
        'F1-Score':    f"{r['f1']:.3f}",
        'ROC-AUC':     f"{r['roc_auc']:.3f}",
        'CV F1':       f"{r['cv_f1']:.3f}±{r['cv_std']:.3f}"
    } for n, r in results.items()
}).T

print('\n=== COMPARAISON DES MODÈLES ===')
print(comp)

In [ ]:
# TODO : Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Resté', 'Churné'],
                yticklabels=['Resté', 'Churné'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'{name}\nF1={r["f1"]:.3f}  AUC={r["roc_auc"]:.3f}', fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.suptitle('Matrices de Confusion — Churn Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TODO : Courbes ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_roc = [COLORS['blue'], COLORS['no_churn'], COLORS['purple']]
for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{name} (AUC={r['roc_auc']:.3f})")

axes[0].plot([0,1],[0,1], 'k--', lw=1, label='Aléatoire')
axes[0].set_xlabel('Taux de faux positifs (FPR)')
axes[0].set_ylabel('Taux de vrais positifs (TPR)')
axes[0].set_title('Courbes ROC', fontweight='bold')
axes[0].legend(loc='lower right')

# Bar chart des métriques
model_names = list(results.keys())
x = np.arange(len(model_names))
w = 0.2
for i, (metric, col) in enumerate([('f1','#185FA5'), ('roc_auc','#1D9E75'), ('recall','#E24B4A')]):
    vals = [results[n][metric] for n in model_names]
    bars = axes[1].bar(x + i*w, vals, w, label=metric.upper(), color=col, edgecolor='white')
    for b, v in zip(bars, vals):
        axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                     f'{v:.2f}', ha='center', fontsize=8)

axes[1].set_xticks(x + w)
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Comparaison des métriques', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 🏆 Section 5 : Best Model Analysis

In [ ]:
# TODO : Analyse du meilleur modèle
best_name = max(results, key=lambda n: results[n]['roc_auc'])
best      = results[best_name]
print(f'🏆 Meilleur modèle : {best_name} (AUC={best["roc_auc"]:.4f})')
print(f'\nRapport de classification complet :')
print(classification_report(y_test, best['y_pred'], target_names=['Resté', 'Churné']))

In [ ]:
# TODO : Importance des features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model_name in zip(axes, ['Random Forest', 'XGBoost']):
    model = results[model_name]['model']
    importances = pd.Series(model.feature_importances_, index=X.columns)
    top10 = importances.sort_values(ascending=True).tail(10)

    colors_bar = ['#1D9E75' if v > top10.mean() else '#B5D4F4' for v in top10.values]
    ax.barh(top10.index, top10.values, color=colors_bar, edgecolor='white')
    ax.set_title(f'Feature Importance — {model_name}', fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Top 10 Features les plus prédictives du churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top features RF
rf_imp = pd.Series(results['Random Forest']['model'].feature_importances_,
                   index=X.columns).sort_values(ascending=False)
print('\n📊 Top 10 features (Random Forest) :')
for feat, imp in rf_imp.head(10).items():
    bar = '█' * int(imp * 200)
    print(f'  {feat:25s} : {imp:.4f} {bar}')

In [ ]:
# TODO : Profils des clients churners vs non-churners
profile_cols = ['Age', 'Balance', 'CreditScore', 'NumOfProducts',
                'IsActiveMember', 'Tenure', 'EstimatedSalary']
profile_cols = [c for c in profile_cols if c in df_clean.columns]

profile = df_clean.groupby('Exited')[profile_cols].mean().T
profile.columns = ['Resté (0)', 'Churné (1)']
profile['Différence (%)'] = ((profile['Churné (1)'] - profile['Resté (0)'])
                              / profile['Resté (0)'] * 100).round(1)

print('=== PROFIL MOYEN : CHURNÉ vs RESTÉ ===')
print(profile.round(2))

# Radar chart
fig, ax = plt.subplots(figsize=(10, 5))
profile_norm = (profile[['Resté (0)', 'Churné (1)']] -
                profile[['Resté (0)', 'Churné (1)']].min()) / \
               (profile[['Resté (0)', 'Churné (1)']].max() -
                profile[['Resté (0)', 'Churné (1)']].min())

x = np.arange(len(profile_norm))
w = 0.35
ax.bar(x - w/2, profile_norm['Resté (0)'], w, label='Resté',
       color=COLORS['no_churn'], edgecolor='white')
ax.bar(x + w/2, profile_norm['Churné (1)'], w, label='Churné',
       color=COLORS['churn'], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(profile_norm.index, rotation=30, ha='right')
ax.set_title('Profil normalisé : Churné vs Resté', fontweight='bold')
ax.set_ylabel('Valeur normalisée (0-1)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 💡 Section 6 : Business Insights & Recommendations

In [ ]:
# TODO : Insights business
print('=' * 65)
print('INSIGHTS BUSINESS — CHURN PREDICTION')
print('=' * 65)

churn_rate = df['Exited'].mean() * 100
print(f'\n📊 Taux de churn global : {churn_rate:.1f}%')

print(f'\n🏆 Meilleur modèle : {best_name}')
print(f'   AUC-ROC  : {best["roc_auc"]:.4f}')
print(f'   F1-Score : {best["f1"]:.4f}')
print(f'   Recall   : {best["recall"]:.4f}  ← % clients churners correctement détectés')
print(f'   Precision: {best["precision"]:.4f} ← % alertes qui sont réels churners')

# Segments à risque
print('\n🔴 SEGMENTS À HAUT RISQUE DE CHURN :')
if 'Age' in df.columns:
    age_churn = df.groupby(pd.cut(df['Age'], bins=[0,30,45,60,100]))['Exited'].mean()
    print(f'  Âge : {age_churn.idxmax()} → taux churn {age_churn.max()*100:.1f}%')
if 'NumOfProducts' in df.columns:
    prod_churn = df.groupby('NumOfProducts')['Exited'].mean()
    print(f'  Nb produits {prod_churn.idxmax()} → taux churn {prod_churn.max()*100:.1f}%')
if 'IsActiveMember' in df.columns:
    active_churn = df.groupby('IsActiveMember')['Exited'].mean()
    print(f'  Clients inactifs → taux churn {active_churn[0]*100:.1f}%')

print('\n✅ RECOMMANDATIONS OPÉRATIONNELLES :')
recs = [
    ('Cibler les clients inactifs',
     'Campagnes de réengagement (email, offres personnalisées)'),
    ('Surveiller les clients 45-60 ans',
     'Offrir des produits adaptés à leur profil (épargne, assurance)'),
    ('Clients avec 3+ produits',
     'Vérifier la satisfaction, simplifier leur expérience bancaire'),
    ('Balance à zéro',
     'Contact proactif, offres de bonus de dépôt'),
    ('Score de crédit faible',
     'Accompagnement financier, coaching budgétaire'),
    ('Déploiement du modèle',
     'Scoring mensuel de chaque client → file de priorité pour les conseillers'),
]
for title, action in recs:
    print(f'  → {title}')
    print(f'     💬 {action}')

In [ ]:
# TODO : Dashboard final de synthèse
fig = plt.figure(figsize=(18, 10))

# 1. Churn rate
ax1 = fig.add_subplot(2, 4, 1)
vals = df['Exited'].value_counts()
ax1.pie(vals, labels=['Resté', 'Churné'], autopct='%1.1f%%',
        colors=[COLORS['no_churn'], COLORS['churn']],
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Répartition Churn', fontweight='bold')

# 2. ROC curve du meilleur modèle
ax2 = fig.add_subplot(2, 4, 2)
fpr, tpr, _ = roc_curve(y_test, best['y_prob'])
ax2.plot(fpr, tpr, color=COLORS['blue'], lw=2,
         label=f'AUC={best["roc_auc"]:.3f}')
ax2.plot([0,1],[0,1],'k--',lw=1)
ax2.set_title(f'ROC — {best_name}', fontweight='bold')
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.legend()

# 3. Feature importance top 5
ax3 = fig.add_subplot(2, 4, 3)
rf_top5 = rf_imp.head(5).sort_values()
ax3.barh(rf_top5.index, rf_top5.values,
         color=COLORS['no_churn'], edgecolor='white')
ax3.set_title('Top 5 Features (RF)', fontweight='bold')

# 4. Age vs churn
ax4 = fig.add_subplot(2, 4, 4)
df[df['Exited']==0]['Age'].plot.hist(ax=ax4, bins=20, alpha=0.6,
    color=COLORS['no_churn'], label='Resté', density=True)
df[df['Exited']==1]['Age'].plot.hist(ax=ax4, bins=20, alpha=0.6,
    color=COLORS['churn'], label='Churné', density=True)
ax4.set_title('Âge par Churn', fontweight='bold')
ax4.legend()

# 5. Comparaison modèles AUC
ax5 = fig.add_subplot(2, 4, 5)
names_short = [n.replace(' ', '\n') for n in results.keys()]
aucs = [r['roc_auc'] for r in results.values()]
bars = ax5.bar(names_short, aucs,
               color=[COLORS['blue'], COLORS['no_churn'], COLORS['purple']],
               edgecolor='white')
ax5.set_ylim(0.5, 1.0)
ax5.set_title('ROC-AUC par modèle', fontweight='bold')
for b, v in zip(bars, aucs):
    ax5.text(b.get_x()+b.get_width()/2, b.get_height()-0.03,
             f'{v:.3f}', ha='center', color='white', fontweight='bold')

# 6. Recall comparé
ax6 = fig.add_subplot(2, 4, 6)
recalls = [r['recall'] for r in results.values()]
bars2 = ax6.bar(names_short, recalls,
                color=[COLORS['blue'], COLORS['no_churn'], COLORS['purple']],
                edgecolor='white')
ax6.set_ylim(0, 1.1)
ax6.set_title('Recall (détection churners)', fontweight='bold')
for b, v in zip(bars2, recalls):
    ax6.text(b.get_x()+b.get_width()/2, b.get_height()-0.05,
             f'{v:.3f}', ha='center', color='white', fontweight='bold')

# 7. Confusion matrix best model
ax7 = fig.add_subplot(2, 4, 7)
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax7,
            xticklabels=['Resté', 'Churné'],
            yticklabels=['Resté', 'Churné'])
ax7.set_title(f'Conf. Matrix — {best_name}', fontweight='bold')

# 8. Products vs Churn
ax8 = fig.add_subplot(2, 4, 8)
ct = pd.crosstab(df['NumOfProducts'], df['Exited'])
ct.plot(kind='bar', ax=ax8, color=[COLORS['no_churn'], COLORS['churn']],
        edgecolor='white', stacked=True)
ax8.set_title('Nb Produits vs Churn', fontweight='bold')
ax8.tick_params(axis='x', rotation=0)
ax8.legend(['Resté', 'Churné'], loc='upper right')

plt.suptitle('🏦 Dashboard — Bank Customer Churn Prediction', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 📋 Conclusion

| Étape | Résultat |
|---|---|
| Dataset | 10 000 clients, 13 variables |
| Taux de churn | ~20% (déséquilibre traité par SMOTE) |
| Meilleur modèle | XGBoost / Random Forest |
| Métrique clé | ROC-AUC > 0.85, Recall élevé |
| Feature la + importante | Age, IsActiveMember, Balance |
| Recommandation | Cibler les inactifs 45-60 ans avec balance zéro |

> **Note métier :** Dans un contexte de rétention bancaire, le **Recall** est la métrique prioritaire — mieux vaut contacter inutilement un bon client (faux positif) que de rater un vrai churner (faux négatif).

---
*Notebook réalisé dans le cadre du Daily Challenge — DI Bootcamp*